# Certainty Evidence Tree: Complete Feature Demo

## What is an Evidence Tree?

An **evidence tree** is a hierarchical proof structure that explains *why* a candidate
was derived. Each node has a `node_kind` from the provenance-role taxonomy:

```
candidate_result          <- the derived fact (root)
├── support_section       <- what evidence supports it?
│   ├── non_fact_check    <- structural checks (ruleref, eq, etc.)
│   └── non_fact_check
└── rule_ref_section      <- which child rules were invoked?
    └── rule_ref          <- pointer to child rule
        └── referenced_support    <- the child proof subtree
            └── support_section
                ├── predicate_witness_group  <- matching facts (carries condition_confidence)
                │   └── assertion_fact       <- the actual witness fact (carries confidence)
                └── predicate_witness_group
                    └── assertion_fact
```

## What is Certainty Propagation?

When facts are written with `confidence` and a rule declares `condition_weights`,
certainty measures how well each condition in the child proof is satisfied:

- Each **assertion** carries a `confidence` from the instance data (e.g., 0.95)
- Each **witness group** aggregates `condition_confidence = max(child confidences)` — tree is **carrier**, not scorer
- **Two aggregation strategies** compute the overall certainty differently:

| Strategy | Impact per condition | Aggregate | Use case |
|----------|---------------------|-----------|----------|
| **bottleneck** (default) | `weight × confidence` | `min(impacts)` | Weakest link determines strength |
| **additive** | `(weight/Σweights) × confidence` | `sum(impacts)` | Each condition contributes proportionally |

## Features Demonstrated

1. Rule `condition_weights` + fact-level `confidence` via `meta`
2. Auto certainty routing (`CertaintyConfidenceKindResolver`)
3. Evidence tree with `confidence` on assertions + `condition_confidence` on witness groups
4. Tree summary (12-field deterministic summary)
5. **Bottleneck** aggregation: `impact = weight × confidence`, `aggregate = min`
6. **Additive** aggregation: `impact = (w/Σw) × confidence`, `aggregate = sum`
7. Narrative with ranked `certainty_lines` + `certainty_bottleneck`
8. NL with weakest-condition sentence
9. Audit round-trip (tree + certainty parity)
10. Static HTML site with certainty section

## 0. Imports

In [1]:
import sys
from pathlib import Path

_src = str(Path("__file__").resolve().parent.parent / "src") if "__file__" in dir() else str(Path.cwd().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)
print("src path:", _src)

src path: /Users/zhenzhili/hnsm-backend/src


In [2]:
from __future__ import annotations

import json
import tempfile
from pathlib import Path
from pprint import pprint

from factpy_kernel.sdk import (
    SDKStore, Entity, Identity, Field, Rule, Pred, vars as sdk_vars,
)
from factpy_kernel.authoring import FileAuthoringRegistry
from factpy_kernel.audit import AuditQuery, load_audit_package, render_audit_static_site
from factpy_kernel.service.runtime_v1 import (
    open_runtime_session, close_runtime_session, reset_runtime_sessions_for_tests,
    write_runtime_fact, evaluate_runtime_derivation, accept_runtime_derivation,
    explain_runtime_tree, explain_runtime_summary,
    explain_runtime_narrative, explain_runtime_nl, export_runtime_package,
)

## 1. Schema & Rule with `condition_weights`

A `User` entity with `tag` and `score` fields. The child rule has **two predicates**
with different weights. We write each fact with explicit **confidence**.

- `b0.a0` → `user:tag` — weight **0.9**, confidence **0.95**
- `b0.a1` → `user:score` — weight **0.4**, confidence **0.6**

In [3]:
class User(Entity):
    user_id: str = Identity(primary_key=True)
    locale: str = Identity()
    name: str = Field(cardinality="single")
    tag: str = Field(cardinality="multi")
    score: str = Field(cardinality="single")

sdk = SDKStore([User])

with sdk_vars("u", "tag", "score") as (u, tag, score):
    child_rule = Rule(
        id="q.qualified_user", version="1.0.0",
        select=[u, tag],
        where=[Pred("user:tag", u, tag), Pred("user:score", u, score)],
        expose=True,
        condition_weights={"b0.a0": 0.9, "b0.a1": 0.4},
    )

print("Rule:", child_rule.id)
print("Condition weights:", child_rule.condition_weights)

Rule: q.qualified_user
Condition weights: {'b0.a0': 0.9, 'b0.a1': 0.4}


## 2. Registry Setup & Seed Data with `confidence`

In [4]:
registry_dir = tempfile.mkdtemp(prefix="certainty_demo_")
registry = FileAuthoringRegistry(Path(registry_dir))
registry.upsert_schema_ir(sdk.schema_ir)
registry.register_rule_spec(sdk._compile_rule_input(child_rule))

with sdk.batch() as tx:
    u1 = tx.entity(User, user_id="u-001", locale="en")
    u1.name.set("Alice")
    u1.tag.add("vip", meta={"confidence": 0.95})
    u1.score.set("85", meta={"confidence": 0.6})
    tx.commit()

u1_ref = sdk.ref(User, user_id="u-001", locale="en")
print("Facts: tag='vip' (conf=0.95), score='85' (conf=0.6)")

Facts: tag='vip' (conf=0.95), score='85' (conf=0.6)


## 3. Evaluate with Auto Certainty Routing

The `CertaintyConfidenceKindResolver` checks at **candidate creation time**:
single resolved `rule_ref_edge` + `condition_weights` in rule payload → `confidence_kind="certainty"`.

In [5]:
reset_runtime_sessions_for_tests()
session_resp = open_runtime_session({"registry_root": registry_dir})
session_id = session_resp["session"]["session_id"]

write_runtime_fact(session_id, {"pred_id": "user:tag", "e_ref": u1_ref, "rest_terms": [["string", "vip"]], "meta": {"confidence": 0.95}}, kind="add")
write_runtime_fact(session_id, {"pred_id": "user:score", "e_ref": u1_ref, "rest_terms": [["string", "85"]], "meta": {"confidence": 0.6}}, kind="add")

eval_resp = evaluate_runtime_derivation(session_id, {
    "derivation": {
        "derivation_id": "drv.certainty_demo", "version": "1.0.0",
        "target": "user:tag", "head_vars": ["$u", "$tag"],
        "where": [["ruleref", "q.qualified_user", "1.0.0", ["$u", "$tag"]], ["eq", "$tag", "vip"]],
        "mode": "native",
    }
})

candidate = eval_resp["evaluation"]["candidates"][0]
candidate_id = candidate["candidate_id"]
print("confidence_kind:", candidate["confidence_kind"], "<- auto-routed at create time")

confidence_kind: certainty <- auto-routed at create time


## 4. Evidence Tree: Confidence Carrier

The tree carries confidence at two levels:
- `assertion_fact.confidence` — raw fact confidence from `meta`
- `predicate_witness_group.condition_confidence` — `max(child confidences)`

In [6]:
tree_resp = explain_runtime_tree(session_id, {"kind": "candidate", "id": candidate_id})
tree = tree_resp["tree"]

def print_tree(node, indent=0):
    prefix = "  " * indent
    kind = node.get("node_kind", "?")
    label = kind
    if kind == "candidate_result":
        label += f"  (root_result_kind={node.get('root_result_kind', '?')})"
    elif kind == "support_section":
        label += f"  ({len(node.get('children', []))} children)"
    elif kind == "rule_ref_section":
        label += f"  ({len(node.get('children', []))} refs)"
    elif kind == "predicate_witness_group":
        label += f"  pred_id={node.get('pred_id', '?')}"
        cc = node.get('condition_confidence')
        if cc is not None:
            label += f"  condition_confidence={cc}"
    elif kind == "assertion_fact":
        claims = node.get("claim_args", [])
        label += f"  [{', '.join(c.get('val','?') for c in claims)}]"
        conf = node.get("confidence")
        if conf is not None:
            label += f"  confidence={conf}"
    elif kind == "non_fact_check":
        label += f"  {node.get('check_kind', '?')}  status={node.get('status', '?')}"
    elif kind == "rule_ref":
        label += f"  {node.get('rule_ref_id', '?')} v{node.get('rule_ref_version', '?')}"
    elif kind == "referenced_support":
        label += f"  digest={node.get('support_digest', '?')[:24]}..."
    print(f"{prefix}- {label}")
    for child in node.get("children", []):
        print_tree(child, indent + 1)

print("=== Evidence Tree ===")
print_tree(tree["root"])

=== Evidence Tree ===
- candidate_result  (root_result_kind=fact)
  - support_section  (2 children)
    - non_fact_check  ruleref  status=satisfied
    - non_fact_check  eq  status=satisfied
  - rule_ref_section  (1 refs)
    - rule_ref  q.qualified_user v1.0.0
      - referenced_support  digest=sha256:483f59e2eaad964c8...
        - support_section  (2 children)
          - predicate_witness_group  pred_id=user:tag  condition_confidence=0.95
            - assertion_fact  [vip]  confidence=0.95
          - predicate_witness_group  pred_id=user:score  condition_confidence=0.6
            - assertion_fact  [85]  confidence=0.6


## 5. Certainty Summary: Bottleneck (Default)

```
b0.a0: weight=0.9 × confidence=0.95 = impact 0.855
b0.a1: weight=0.4 × confidence=0.6  = impact 0.24  ← BOTTLENECK
aggregate = min(0.855, 0.24) = 0.24
```

In [7]:
summary_resp = explain_runtime_summary(session_id, {"kind": "candidate", "id": candidate_id})
cs_bottleneck = summary_resp["certainty_summary"]

print("=== Certainty Summary (bottleneck) ===")
pprint(cs_bottleneck)

=== Certainty Summary (bottleneck) ===
{'aggregate_certainty': 0.24,
 'aggregation': 'bottleneck',
 'condition_count': 2,
 'conditions': [{'atom_key': 'b0.a0',
                 'impact': 0.855,
                 'node_kind': 'predicate_witness_group',
                 'weight': 0.9},
                {'atom_key': 'b0.a1',
                 'impact': 0.24,
                 'node_kind': 'predicate_witness_group',
                 'weight': 0.4}],
 'confidence_kind': 'certainty',
 'weighted_condition_count': 2}


## 6. Certainty Summary: Additive (Second Strategy)

Pass `certainty_aggregation="additive"`. Weights are **normalized**:
```
b0.a0: (0.9/1.3) × 0.95 = 0.658    b0.a1: (0.4/1.3) × 0.6 = 0.185
aggregate = 0.658 + 0.185 = 0.843
```
No bottleneck — all conditions contribute proportionally.

In [8]:
summary_add = explain_runtime_summary(session_id, {
    "kind": "candidate", "id": candidate_id, "certainty_aggregation": "additive",
})
cs_additive = summary_add["certainty_summary"]

print("=== Certainty Summary (additive) ===")
pprint(cs_additive)

print()
print("=== Side-by-Side Comparison ===")
print(f"  {'':20s} {'Bottleneck':>12s}  {'Additive':>12s}")
print(f"  {'Aggregate':20s} {cs_bottleneck['aggregate_certainty']:>12}  {cs_additive['aggregate_certainty']:>12}")
for bn_c, ad_c in zip(cs_bottleneck['conditions'], cs_additive['conditions']):
    print(f"  {bn_c['atom_key']:20s} {bn_c['impact']:>12}  {ad_c['impact']:>12}")
print(f"  {'Strategy':20s} {'min(impacts)':>12s}  {'sum(contribs)':>12s}")

=== Certainty Summary (additive) ===
{'aggregate_certainty': 0.842307,
 'aggregation': 'additive',
 'condition_count': 2,
 'conditions': [{'atom_key': 'b0.a0',
                 'impact': 0.657692,
                 'node_kind': 'predicate_witness_group',
                 'weight': 0.9},
                {'atom_key': 'b0.a1',
                 'impact': 0.184615,
                 'node_kind': 'predicate_witness_group',
                 'weight': 0.4}],
 'confidence_kind': 'certainty',
 'weighted_condition_count': 2}

=== Side-by-Side Comparison ===
                         Bottleneck      Additive
  Aggregate                    0.24      0.842307
  b0.a0                       0.855      0.657692
  b0.a1                        0.24      0.184615
  Strategy             min(impacts)  sum(contribs)


## 7. Narrative: Ranked Conditions (Both Strategies)

Conditions sorted by impact ascending. Bottleneck mode marks `[bottleneck]`.

In [9]:
narr_bn = explain_runtime_narrative(session_id, {"kind": "candidate", "id": candidate_id})["narrative"]
narr_ad = explain_runtime_narrative(session_id, {"kind": "candidate", "id": candidate_id, "certainty_aggregation": "additive"})["narrative"]

print("=== Bottleneck Narrative ===")
for line in narr_bn.get("certainty_lines", []): print(" ", line)
print("  bottleneck:", narr_bn.get("certainty_bottleneck"))

print()
print("=== Additive Narrative ===")
for line in narr_ad.get("certainty_lines", []): print(" ", line)
print("  bottleneck:", narr_ad.get("certainty_bottleneck", "(none)"))

=== Bottleneck Narrative ===
  Certainty (eligible child-proof subtree): aggregate certainty (bottleneck): 0.24.
  Condition b0.a1 (predicate_witness_group): weight=0.4, impact=0.24. [bottleneck]
  Condition b0.a0 (predicate_witness_group): weight=0.9, impact=0.855.
  bottleneck: {'atom_keys': ['b0.a1'], 'impact': 0.24}

=== Additive Narrative ===
  Certainty (eligible child-proof subtree): aggregate certainty (additive): 0.842307.
  Condition b0.a1 (predicate_witness_group): weight=0.4, impact=0.184615.
  Condition b0.a0 (predicate_witness_group): weight=0.9, impact=0.657692.
  bottleneck: (none)


## 8. NL Explain: Weakest-Condition Sentence

The NL layer appends a 5th paragraph with a human-readable certainty summary.

In [10]:
nl_resp = explain_runtime_nl(session_id, {"kind": "candidate", "id": candidate_id})
print("=== NL Explain (5 paragraphs) ===")
for i, para in enumerate(nl_resp["explain_nl"]["paragraphs"]):
    print(f"  [{i+1}] {para}\n")

=== NL Explain (5 paragraphs) ===
  [1] Candidate cand_v2:33935d48cc9d55b0699bffb3adbb3502f7441f970cb7af2b0c30869f5e642b3e uses support kind native_binding_v1 across 12 tree node(s). Root result kind: fact. Role counts: structural=4, witness=4, constraint=2, rule_chain=2, terminal=0, degraded=0. Recursive depth: 1.

  [2] Evidence summary: Witness assertions: 2. Witness nodes: 4; constraint nodes: 2.

  [3] Rule-chain summary: Rule reference nodes: 1. Recursive proof depth: 1.

  [4] Terminal and drill-down summary: No unresolved support or recursion boundaries were encountered. Open referenced support branches to inspect recursive child proof. Open linked assertion nodes to inspect witness facts.

  [5] Certainty summary: Certainty (eligible child-proof subtree): aggregate certainty (bottleneck): 0.24. Condition b0.a1 (predicate_witness_group): weight=0.4, impact=0.24. [bottleneck] Condition b0.a0 (predicate_witness_group): weight=0.9, impact=0.855. The weakest condition is b0.a1 with

## 9. Audit Round-Trip + Static Site

Accept → export audit package (writes `certainty_summaries.jsonl`) → load → verify parity → render static HTML.

In [11]:
accept_runtime_derivation(session_id, {"candidate": candidate, "options": {"approved_by": "demo"}})

import tempfile as _tf
_pkg = _tf.TemporaryDirectory(prefix="pkg_")
_site = _tf.TemporaryDirectory(prefix="site_")

export_runtime_package(session_id, {"out_dir": _pkg.name, "package_kind": "audit"})
package = load_audit_package(_pkg.name)
aq = AuditQuery(package)

audit_tree = aq.get_candidate_evidence_tree(candidate_id)
print("=== Audit Evidence Tree ===")
if audit_tree and "root" in audit_tree: print_tree(audit_tree["root"])

audit_cs = aq.get_candidate_certainty_summary(candidate_id)
print("\n=== Audit Certainty Summary ===")
pprint(audit_cs)

audit_narr = aq.get_candidate_evidence_tree_narrative(candidate_id)
print("\n=== Audit Certainty Lines ===")
if audit_narr:
    for line in audit_narr.get("certainty_lines", []): print(" ", line)

print("\nCertainty parity (runtime == audit):", audit_cs == cs_bottleneck)

render_audit_static_site(_pkg.name, _site.name)
pages = sorted(Path(_site.name).rglob("*.html"))
print(f"\nStatic site: {len(pages)} HTML pages")
for p in pages: print(f"  {p.relative_to(_site.name)}  ({p.stat().st_size} bytes)")

=== Audit Evidence Tree ===
- candidate_result  (root_result_kind=fact)
  - support_section  (2 children)
    - non_fact_check  ruleref  status=satisfied
    - non_fact_check  eq  status=satisfied
  - rule_ref_section  (1 refs)
    - rule_ref  q.qualified_user v1.0.0
      - referenced_support  digest=sha256:483f59e2eaad964c8...
        - support_section  (2 children)
          - predicate_witness_group  pred_id=user:tag
            - assertion_fact  [vip]
          - predicate_witness_group  pred_id=user:score
            - assertion_fact  [85]

=== Audit Certainty Summary ===
{'aggregate_certainty': 0.24,
 'aggregation': 'bottleneck',
 'condition_count': 2,
 'conditions': [{'atom_key': 'b0.a0',
                 'impact': 0.855,
                 'node_kind': 'predicate_witness_group',
                 'weight': 0.9},
                {'atom_key': 'b0.a1',
                 'impact': 0.24,
                 'node_kind': 'predicate_witness_group',
                 'weight': 0.4}],
 'confid

### 9b. Static Site: Candidate Evidence HTML

In [12]:
from IPython.display import HTML, display
import glob

candidate_pages = glob.glob(f"{_site.name}/candidate_evidence/*.html")
if candidate_pages:
    page_path = candidate_pages[0]
    html_content = Path(page_path).read_text(encoding="utf-8")
    print(f"Displaying: {Path(page_path).name} ({len(html_content)} chars)")
    display(HTML(f"<div style='border:2px solid #ccc;border-radius:8px;overflow:hidden;margin:10px 0'><div style='background:#f5f5f5;padding:8px 12px;border-bottom:1px solid #ccc;font-family:monospace;font-size:12px'>{Path(page_path).name}</div><iframe srcdoc='{html_content.replace(chr(39), chr(38)+chr(35)+chr(51)+chr(57)+chr(59))}' style='width:100%;height:600px;border:none'></iframe></div>"))


Displaying: cand_v2%3A33935d48cc9d55b0699bffb3adbb3502f7441f970cb7af2b0c30869f5e642b3e.html (22176 chars)


## 10. Negative Case: No `condition_weights` → No Certainty

Without `condition_weights`, the resolver falls back to `"none"` — even with confidence on facts.

In [13]:
with sdk_vars("u", "tag", "score") as (u, tag, score):
    plain_rule = Rule(id="q.plain_rule", version="1.0.0", select=[u, tag],
        where=[Pred("user:tag", u, tag), Pred("user:score", u, score)], expose=True)
registry.register_rule_spec(sdk._compile_rule_input(plain_rule))

eval2 = evaluate_runtime_derivation(session_id, {"derivation": {
    "derivation_id": "drv.plain", "version": "1.0.0", "target": "user:tag",
    "head_vars": ["$u", "$tag"],
    "where": [["ruleref", "q.plain_rule", "1.0.0", ["$u", "$tag"]], ["eq", "$tag", "vip"]],
    "mode": "native",
}})
pc = eval2["evaluation"]["candidates"][0]
ps = explain_runtime_summary(session_id, {"kind": "candidate", "id": pc["candidate_id"]})
print(f"confidence_kind: {pc['confidence_kind']}")
print(f"certainty_summary: {ps.get('certainty_summary')}")
print("\n>>> No condition_weights -> no certainty routing -> no certainty delivery")

confidence_kind: none
certainty_summary: None

>>> No condition_weights -> no certainty routing -> no certainty delivery


In [14]:
close_runtime_session(session_id)
reset_runtime_sessions_for_tests()
_pkg.cleanup(); _site.cleanup()
print("Done.")

Done.


## Feature Summary

| Feature | What happens |
|---------|-------------|
| **Rule condition_weights** | `{"b0.a0": 0.9, "b0.a1": 0.4}` declared on child rule |
| **Fact confidence** | `meta={"confidence": 0.95}` written with each fact |
| **Auto routing** | `CertaintyConfidenceKindResolver` → `confidence_kind="certainty"` at create time |
| **Tree carrier** | `assertion_fact.confidence` + `predicate_witness_group.condition_confidence=max(children)` |
| **Tree summary** | 12-field deterministic summary |
| **Bottleneck** | `impact = weight × confidence`, `aggregate = min(impacts)` — default |
| **Additive** | `impact = (w/Σw) × confidence`, `aggregate = sum(impacts)` — via `certainty_aggregation` |
| **Narrative** | Ranked `certainty_lines`, `[bottleneck]` marker, `certainty_bottleneck` key |
| **NL** | 5th paragraph: weakest-condition sentence |
| **Audit export** | `certainty_summaries.jsonl` materialized at export time |
| **Audit query** | `get_candidate_certainty_summary()` + narrative with certainty_lines |
| **Static HTML** | Candidate evidence pages with certainty section |
| **Negative case** | No `condition_weights` → `"none"`, no certainty delivery |